# Experimentos de ranking

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent  # if notebook is in notebooks/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.paths import PROC_DIR, ARTIFACTS_DIR
from src.retrieval import BM25Retriever
from src.evaluation import map_relevance, evaluate_retrieval, evaluate_ranking
from src.features import build_ranking_features
from src.ranking import train_ranker, score_candidates, save_ranker, load_ranker
from src.evaluation import evaluate_ranking


In [3]:
TOP_K = 100

In [4]:
MODEL_PATH = ARTIFACTS_DIR / "hgb_ranker.joblib"

In [5]:
products = pd.read_parquet(PROC_DIR / "products.parquet")
queries = pd.read_parquet(PROC_DIR / "queries.parquet")
labels = pd.read_parquet(PROC_DIR / "labels.parquet")
splits = pd.read_parquet(PROC_DIR / "splits.parquet")

In [6]:
labels = labels.copy()
labels["relevance"] = map_relevance(labels["label"])

train_qids = set(splits.loc[splits["split"] == "train", "query_id"])
val_qids = set(splits.loc[splits["split"] == "val", "query_id"])

train_queries = queries[queries["query_id"].isin(train_qids)]
val_queries = queries[queries["query_id"].isin(val_qids)]

train_labels = labels[labels["query_id"].isin(train_qids)]
val_labels = labels[labels["query_id"].isin(val_qids)]

In [7]:
print(f"Train queries: {len(train_queries)} | Val queries: {len(val_queries)}")

Train queries: 336 | Val queries: 72


### BM25

In [8]:
retriever = BM25Retriever().fit(products)
train_candidates = retriever.retrieve(train_queries, top_k=TOP_K)
val_candidates = retriever.retrieve(val_queries, top_k=TOP_K)

In [9]:
retrieval_metrics = evaluate_retrieval(val_candidates, val_labels, ks=[20, 50, TOP_K])
print("Retrieval metrics (val):", retrieval_metrics)

Retrieval metrics (val): {'recall@20': 0.12425121013626411, 'recall@50': 0.2508804523216268, 'recall@100': 0.3830537592768249}


In [10]:
train_feat = build_ranking_features(
    candidates=train_candidates,
    queries=train_queries,
    products=products,
    labels=train_labels,
)

In [11]:
val_feat = build_ranking_features(
    candidates=val_candidates,
    queries=val_queries,
    products=products,
    labels=val_labels,
)

In [12]:
print("Train features shape:", train_feat.shape, "| Val features shape:", val_feat.shape)

Train features shape: (33600, 31) | Val features shape: (7200, 31)


In [13]:
bm25_val_scored = val_feat[["query_id", "bm25_score", "relevance"]].rename(
    columns={"bm25_score": "score"}
)
bm25_val_metrics = evaluate_ranking(bm25_val_scored, ks=[10])

In [14]:
bm25_val_metrics

{'ndcg@10': 0.7102650918215304}

### HistGradientBoostingRegressor

In [15]:
ranker, feat_cols = train_ranker(
    train_df=train_feat,
    val_df=val_feat,
    params={
        "loss": "squared_error",
        "learning_rate": 0.05,
        "max_iter": 250,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 20,
        "random_state": 42,
    },
)

In [16]:
val_scored = score_candidates(ranker, val_feat, feat_cols)

In [17]:
ranker_val_metrics = evaluate_ranking(val_scored, ks=[10])

In [18]:
comparison_df = pd.DataFrame(
    [
        {"model": "BM25 Baseline", **bm25_val_metrics},
        {"model": "HistGradientBoosting", **ranker_val_metrics},
    ]
)

In [19]:
comparison_df

,model,ndcg@10
0,BM25 Baseline,0.710265
1,HistGradientBoosting,0.775547


### Salvando modelo

In [20]:
saved_path = save_ranker(ranker, feat_cols, MODEL_PATH)
reloaded_model, reloaded_cols = load_ranker(saved_path)
reloaded_scored = score_candidates(reloaded_model, val_feat, reloaded_cols)

assert np.allclose(val_scored["score"].to_numpy(), reloaded_scored["score"].to_numpy())
print(f"Saved and verified ranker at: {saved_path} ({len(reloaded_cols)} features)")

Saved and verified ranker at: C:\Users\kadub\OneDrive\Desktop\projeto-busca-ranking\artifacts\hgb_ranker.joblib (28 features)


### Feature-group ablation

In [22]:
from src.ranking import get_feature_columns

In [23]:
all_cols = get_feature_columns(train_feat)

In [24]:
retrieval_cols = [c for c in all_cols if "bm25" in c or "rank" in c or "overlap" in c or "match" in c]
retrieval_cols

['rank',
 'bm25_score',
 'retrieval_rank',
 'reciprocal_rank',
 'bm25_diff_from_max',
 'bm25_diff_from_mean',
 'name_token_overlap',
 'name_exact_query_match',
 'class_token_overlap',
 'class_exact_query_match',
 'cat_token_overlap',
 'cat_exact_query_match']

In [25]:
numeric_cols = [c for c in all_cols if c not in retrieval_cols]
numeric_cols

['score',
 'average_rating',
 'rating_count',
 'review_count',
 'has_rating',
 'log1p_rating_count',
 'log1p_review_count',
 'name_char_len',
 'desc_char_len',
 'query_token_count',
 'name_query_coverage',
 'name_jaccard',
 'class_query_coverage',
 'class_jaccard',
 'cat_query_coverage',
 'cat_jaccard']

In [26]:
ablation_sets = {
    "BM25 only (baseline)": None,
    "Retrieval / lexical features": retrieval_cols,
    "Product numeric features": numeric_cols,
    "All features (BM25 + lexical + numeric)": all_cols,
}

In [27]:
rows = []
for name, cols in ablation_sets.items():
    if cols is None:
        scored = val_feat[["query_id", "bm25_score", "relevance"]].rename(columns={"bm25_score": "score"})
    else:
        model, used_cols = train_ranker(train_feat, val_df=val_feat, feature_cols=cols)
        scored = score_candidates(model, val_feat, used_cols)
    metrics = evaluate_ranking(scored, ks=[10])
    rows.append({"feature_set": name, "n_features": 0 if cols is None else len(cols), **metrics})

In [28]:
print(pd.DataFrame(rows).to_string(index=False))

                            feature_set  n_features  ndcg@10
                   BM25 only (baseline)           0 0.710265
           Retrieval / lexical features          12 0.767514
               Product numeric features          16 0.768563
All features (BM25 + lexical + numeric)          28 0.775547


### One-shot evaluation

In [29]:
test_qids = set(splits.loc[splits["split"] == "test", "query_id"])
assert test_qids.isdisjoint(train_qids), "Leakage: test queries overlap with train"
assert test_qids.isdisjoint(val_qids), "Leakage: test queries overlap with val"

In [30]:
test_queries = queries[queries["query_id"].isin(test_qids)]
test_labels = labels[labels["query_id"].isin(test_qids)]

- Retrieval on held-out test set

In [31]:
test_candidates = retriever.retrieve(test_queries, top_k=TOP_K)
test_retrieval_metrics = evaluate_retrieval(test_candidates, test_labels, ks=[20, 50, TOP_K])
print("Test Retrieval Metrics:", test_retrieval_metrics)

Test Retrieval Metrics: {'recall@20': 0.09691860584905647, 'recall@50': 0.23084486794880174, 'recall@100': 0.3661810483848724}


- Build features and score with saved ranker (no refitting)

In [32]:
test_feat = build_ranking_features(
    candidates=test_candidates,
    queries=test_queries,
    products=products,
    labels=test_labels,
)

In [33]:
ranker, feat_cols = load_ranker(ARTIFACTS_DIR / "hgb_ranker.joblib")
test_scored = score_candidates(ranker, test_feat, feat_cols)

- Compare BM25 vs. HistGradientBoosting NDCG@10 on test

In [34]:
bm25_test_scored = test_feat[["query_id", "bm25_score", "relevance"]].rename(
    columns={"bm25_score": "score"}
)

In [35]:
bm25_test_metrics = evaluate_ranking(bm25_test_scored, ks=[10])

In [36]:
ranker_test_metrics = evaluate_ranking(test_scored, ks=[10])

In [37]:
test_summary_df = pd.DataFrame(
    [
        {"model": "BM25 Baseline", **bm25_test_metrics},
        {"model": "HistGradientBoosting", **ranker_test_metrics},
    ]
)

In [38]:
print(f"\nTest queries: {len(test_queries)} | Test candidate rows: {len(test_feat)}")
print(test_summary_df.to_string(index=False))


Test queries: 72 | Test candidate rows: 7200
               model  ndcg@10
       BM25 Baseline 0.670603
HistGradientBoosting 0.737970
